In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
### Simple Gen AI App using Lanchain

from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
loader

In [ ]:
docs = loader.load()
print(len(docs))
docs

1


[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, t

Load Data -->-> Docs ->-> Devide Text into Chunks ->-> Vectors ->-> Vector Embeddings ->-> Vector Store DB

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)
print(len(documents))
documents

4


[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, t

In [ ]:
from langchain_openai import ChatOpenAIEmbeddings
embeddings = ChatOpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
from langchain_community.vectorstores import FAISS

vectorestoredb = FAISS.from_documents(documents=documents, embedding=embeddings)
vectorestoredb

NameError: name 'embeddings' is not defined

In [ ]:
## query from vectore store db
query = "which determine their access to resources in that workspace?"
result = vectorestoredb.similarity_search(query)
result[0].page_content

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
## retrieval chain , document chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    """
    Answer the question based on the following context:
    
    <context>
    {context}
    </context>
    
    """
)

document_chain = create_stuff_documents_chain(llm, prompt=prompt)
document_chain

In [ ]:
from langchain_core.documents import Document

document_chain.invoke(
    {
        "input": "Langsmith has two usages limits: total traces and extended ",
        "context": [ Document(page_content="Langsmith has two usages limits: \
                            total traces and extended A workspace is a logical grouping of users and resources within an organization.")]
    }
)

In [ ]:
## Retriever => It can be consiter as interface,
# retriever is responsible for fetching relevant documents based on a query,
#  while the document chain processes those documents to generate a response. By using a retriever, 
# you can separate the concerns of retrieving relevant information and processing that information to answer questions or generate content.
retriever=vectorestoredb.as_retriever()

from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain) # document chain is responsible for context information

In [ ]:
## Get the response from the llm
# here input is from user
response = retrieval_chain.invoke(
    {
        "input": "Langsmith has two usages limits: total traces and extended "
    }
)
response["answer"]

In [ ]:
response

In [ ]:
response['context']